# 04 — Uji Awal Baseline PostFilter (B0)

**Tahap 7 ceklis: Implementasi metode** · README bagian 18 (B0), 24.4, 24.5, 25

**Status: uji coba pengembangan, bukan eksperimen resmi.** Himpunan kueri belum dibekukan (Tahap 5), sehingga kueri di sini adalah dokumen korpus yang dipilih acak.

**Cara kerja PostFilter:** penelusuran graf **tanpa filter** dengan daftar kandidat L = 100 (`candidate_budget`), lalu predikat diterapkan pada 100 kandidat itu dan diambil 10 teratas yang lolos. Bila kandidat yang lolos kurang dari 10, hasilnya *under-filled*.

**Implementasi:** graf DiskANN yang sama (`index/diskann/tourism`) dibaca oleh `src/search/graph_index.py`; penelusurannya ditulis ulang di Python karena `diskannpy` hanya menyediakan `search`/`batch_search`. Pada uji 200 kueri, hasil top-10 penelusuran Python identik dengan `diskannpy`.

## Langkah 1: Membaca vektor, graf, dan korpus

In [1]:
# Vektor beku, graf DiskANN bersama, dan korpus beserta metadata
import numpy as np
import pandas as pd
import sys
sys.path.append("..")

from src.search.graph_index import load_vamana_graph, greedy_search
from src.search.postfilter import postfilter_search
from src.groundtruth.exact_filtered_knn import exact_filtered_knn

vectors = np.load("../data/embeddings/base_vectors.npy")
start, neighbors = load_vamana_graph("../index/diskann/tourism")
corpus = pd.read_parquet("../data/processed/corpus.parquet")

print("Titik awal graf:", start, "| jumlah titik:", len(neighbors))

Titik awal graf: 1988 | jumlah titik: 64354


## Langkah 2: Kesetaraan penelusuran Python dengan DiskANN

Recall@10 tanpa filter pada 200 dokumen acak (dokumen kueri sendiri dibuang dari hasil). Angka ini harus mendekati uji kewarasan saat build (0,9959).

In [2]:
# Recall@10 tanpa filter dari penelusuran Python pada 200 dokumen acak (dokumen itu sendiri dibuang)
rng = np.random.default_rng(2906)
sampel = rng.choice(len(vectors), size=200, replace=False)

recall = []
for i in sampel:
    pool, _ = greedy_search(vectors[i], vectors, neighbors, start, L=100)
    ann = [j for _, j in pool if j != i][:10]
    eksak = [j for j in np.argsort(-(vectors @ vectors[i])) if j != i][:10]
    recall.append(len(set(ann) & set(eksak)) / 10)

print("Recall@10 tanpa filter:", round(np.mean(recall), 4))

Recall@10 tanpa filter: 0.999


## Langkah 3: PostFilter pada predikat dengan selektivitas berbeda

Empat predikat alami dari bin yang berbeda (B5 → B2), masing-masing diuji dengan 100 kueri yang sama. Untuk setiap kueri, dokumen kueri dikeluarkan dari himpunan eligible (mencegah self-match), ground truth dihitung eksak dengan `exact_filtered_knn`, dan setiap hasil diperiksa lolos predikat (`assert`, README bagian 24.4).

In [3]:
# Predikat alami dari korpus, dari yang lebar hingga sangat selektif
predikat = {
    "kategori = alam": corpus["category"].apply(lambda d: "alam" in d).values,
    "provinsi = Bali": corpus["province"].apply(lambda d: "Bali" in d).values,
    "penerbit = kompas.com": (corpus["publisher_id"] == "kompas.com").values,
    "kabupaten = Tanggamus": corpus["kabupaten"].apply(lambda d: "Kabupaten Tanggamus" in d).values,
}

# 100 dokumen acak sebagai kueri; dokumen kueri dikeluarkan dari himpunan eligible (cegah self-match)
kueri_idx = rng.choice(len(vectors), size=100, replace=False)

baris = []
for nama, mask in predikat.items():
    recalls, underfilled, kunjungan = [], 0, []
    for i in kueri_idx:
        eligible = mask.copy()
        eligible[i] = False
        hasil, stats = postfilter_search(vectors[i], vectors, neighbors, start, eligible, k=10, candidate_budget=100)
        gt = exact_filtered_knn(vectors, vectors[i], eligible, k=10)
        recalls.append(len(set(hasil) & set(gt)) / 10)
        underfilled += stats["underfilled"]
        kunjungan.append(stats["visited_nodes"])
        assert all(eligible[j] for j in hasil)   # semua hasil wajib lolos predikat

    baris.append({
        "predikat": nama,
        "match_rate_%": round(mask.mean() * 100, 3),
        "recall@10": round(np.mean(recalls), 3),
        "under-filled": f"{underfilled}/100",
        "visited_nodes": int(np.mean(kunjungan)),
    })

pd.DataFrame(baris)

,predikat,match_rate_%,recall@10,under-filled,visited_nodes
0,kategori = alam,40.771,0.909,15/100,102
1,provinsi = Bali,3.467,0.290,95/100,102
2,penerbit = kompas.com,2.620,0.189,100/100,102
3,kabupaten = Tanggamus,0.171,0.023,100/100,102


## Ringkasan hasil

| Predikat | Match rate | Bin | Recall@10 | Under-filled | Visited nodes |
|---|---:|---|---:|---:|---:|
| kategori = alam | 40,771% | B5 | 0,909 | 15/100 | 102 |
| provinsi = Bali | 3,467% | B4 | 0,290 | 95/100 | 102 |
| penerbit = kompas.com | 2,620% | B4 | 0,189 | 100/100 | 102 |
| kabupaten = Tanggamus | 0,171% | B2 | 0,023 | 100/100 | 102 |

- Penelusuran Python setara dengan DiskANN: recall@10 tanpa filter **0,999**.
- **Recall PostFilter jatuh tajam saat filter makin selektif**: 0,909 → 0,290 → 0,189 → 0,023.
- **Under-filled hampir selalu terjadi di bawah ~3,5%**: 95–100 dari 100 kueri tidak mendapat 10 hasil.
- **Biaya pencarian konstan (102 visited nodes)** untuk semua predikat, karena PostFilter menelusuri graf tanpa memedulikan filter.
- Semua hasil lolos predikat (predicate satisfaction rate 100%).

## Kesimpulan dan implikasi (draf)

1. **Kelemahan PostFilter yang diprediksi paper acuan terbukti pada korpus pariwisata.** Dengan 100 kandidat, peluang kandidat lolos filter sebanding dengan match rate; pada match rate 0,17% hanya sekitar 0–1 kandidat yang lolos. Inilah celah yang ingin ditutup TwoHop dan AdaptiveL.
2. **Biaya konstan adalah akar masalahnya.** PostFilter tidak menambah usaha pencarian saat filter selektif. AdaptiveL justru dirancang memperbesar daftar kandidat ketika match rate yang diestimasi rendah.
3. **Under-filled 15% pada filter lebar (40,8%) menunjukkan korelasi kueri–filter.** Kueri dokumen budaya bertetangga dengan dokumen budaya lain (teks JADESTA memuat "Wisata budaya."), sehingga 100 kandidatnya jarang berkategori alam. Efek korelasi ini perlu dicatat sebagai faktor selain match rate saat menganalisis hasil.
4. **Catatan untuk eksperimen resmi:** QPS dan latency dari penelusuran Python tidak sebanding dengan implementasi C++ DiskANN. Perbandingan antarmetode tetap adil karena semua metode memakai penelusuran Python yang sama, tetapi angka absolutnya tidak boleh dibandingkan dengan paper acuan. Ini perlu dikonfirmasi saat bimbingan.